# Analyse Économique — Réseaux de Chaleur

**Auteur :** Emmanuel TSAGUE — EDF MAD EDVANCE | Datascientest 2024  
**Données :** simulées — 30 projets de réseaux de chaleur

### Plan
1. Portefeuille de projets
2. Métriques économiques (VAN, TRI, temps de retour)
3. Analyse CO₂ et ENR
4. Scoring et priorisation
5. Matrice VAN / TRI
6. Recommandations

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data_generation import load_or_generate
from src.economic_analysis import (npv, irr, payback_period,
                                    project_cash_flows, rank_projects,
                                    plot_portfolio_analysis)
print('Imports OK')

## 1. Portefeuille de Projets

In [ ]:
df = load_or_generate('../data_sample/district_heating_simulated.csv')
print(f'Projets : {len(df)}')
print(f'CAPEX total    : {df.capex_total_keur.sum()/1000:,.0f} M€')
print(f'Revenue annuel : {df.revenue_annuel_meur.sum():.1f} M€/an')
df.head()

## 2. Métriques Économiques

In [ ]:
print('Statistiques économiques :')
print(df[['npv_meur_25ans','irr_pct','payback_ans','taux_enr_pct']].describe().round(2))

In [ ]:
# Exemple de calcul VAN pour un projet
project = df.iloc[0]
flows = project_cash_flows(
    revenue=project['revenue_annuel_meur'],
    opex=project['opex_annuel_meur'],
    capex=project['capex_total_keur'] / 1000,
    years=25
)
van = npv(flows[:-1], discount_rate=0.05, capex=project['capex_total_keur']/1000)
tri = irr(flows[:-1], capex=project['capex_total_keur']/1000)
print(f"Projet : {project['ville']} — {project['source_energie']}")
print(f'VAN 25 ans : {van:.2f} M€')
print(f'TRI        : {tri*100:.2f}%')
print(f'Retour     : {payback_period(project["ebitda_meur"], project["capex_total_keur"]/1000):.1f} ans')

## 3. Analyse ENR et CO₂

In [ ]:
print('Taux ENR moyen par source :')
print(df.groupby('source_energie')['taux_enr_pct'].mean().round(1))
print('\nCO₂ évité total par source (kt) :')
print((df.groupby('source_energie')['co2_evite_t'].sum()/1000).round(1))

## 4. Scoring et Analyse de Portefeuille

In [ ]:
ranked = plot_portfolio_analysis(df)

## 5. Top Projets Viables

In [ ]:
top = ranked[ranked['viable']].head(10)
print('Top projets viables :')
print(top[['ville','source_energie','npv_meur_25ans','irr_pct',
           'payback_ans','taux_enr_pct','co2_evite_t']].to_string(index=False))

## Synthèse

| Critère | Seuil viabilité |
|---------|----------------|
| VAN | > 0 M€ |
| TRI | > 6% |
| Retour | < 20 ans |
| Taux ENR | > 50% |

> **Meilleures sources :** géothermie, biomasse, récupération chaleur  

*Données simulées (seed=42) — Emmanuel TSAGUE*